# RAVE-SIM 2D Simulation Test

Tests both Python (big-wave) and C++ (fast-wave) 2D simulation capabilities.

## Test Scenarios
1. **2D Free-space propagation** — point source → Fresnel propagation in 2D
2. **2D Sample diffraction** — 3D material grid (W sphere) → 2D wavefront → detector
3. **1D PlasmaSample** — plasma delta/beta physics verification (1D only for now)
4. **Generate fast-wave 2D config** — create config file for C++ GPU solver

---

In [ ]:
import sys, os
from pathlib import Path
import warnings
import numpy as np

# add big-wave to path
try:
    rave_sim_dir = Path(get_ipython().run_line_magic('pwd', ''))
except Exception:
    rave_sim_dir = Path.cwd()
    for _ in range(10):
        if (rave_sim_dir / 'big-wave').is_dir():
            break
        rave_sim_dir = rave_sim_dir.parent
rave_sim_dir = rave_sim_dir.resolve()
sys.path.insert(0, str(rave_sim_dir / 'big-wave'))
print(f'RAVE-SIM dir: {rave_sim_dir}')

# ── bfpy shim (for environments without the compiled Rust module) ──
try:
    import bfpy
except ImportError:
    print('bfpy not found - installing minimal shim for NumpyVector-only use')
    import types
    _bfpy = types.ModuleType('bfpy')
    class _CE:
        def __init__(self, *a, **kw): pass
        def position(self): return 0
        def read_chunk_c16(self, b): return len(b)
        def read_chunk_c8(self, b): return len(b)
        def write_chunk_c16(self, b): pass
        def write_chunk_c8(self, b): pass
        def advance(self, n): pass
        def __len__(self): return 0
        def dtype(self): return 'c16'
    def _g(*a, **kw): pass
    _bfpy.ChunkedEditor = _CE
    _bfpy.generate_header_c16 = _g; _bfpy.generate_header_c8 = _g
    _bfpy.fft_c16 = _g; _bfpy.fft_c8 = _g; _bfpy.ifft_c16 = _g; _bfpy.ifft_c8 = _g
    sys.modules['bfpy'] = _bfpy
    print('bfpy shim installed')

# ── Import order matters to avoid circular imports ────────────
import config
from propagation import (
    SimParams, propagate, propagate_2d, propagate_analytically,
    propagate_analytically_2d, square_and_downsample,
    square_and_downsample_2d, convert_energy_wavelength,
    max_dx,
)
from source import PointSource
from plasma_sample import PlasmaSample
from plasma import plasma_delta_beta
from vector import NumpyVector

warnings.filterwarnings('ignore')
print('all imports done')

In [ ]:
# ── helper functions ────────────────────────────────────────

def make_sphere_grid(nx, ny, nz, radius_px, cx=None, cy=None):
    """Create a 3D uint32 grid (nz, ny, nx) with a sphere of material 1."""
    cx = cx or nx // 2
    cy = cy or ny // 2
    grid = np.zeros((nz, ny, nx), dtype=np.uint32)
    zz, yy, xx = np.ogrid[:nz, :ny, :nx]
    mask = (xx - cx)**2 + (yy - cy)**2 + (zz - nz//2)**2 <= radius_px**2
    grid[mask] = 1
    return grid

def make_gaussian_plasma(nz, nx, ne_peak, Z_star, T_e):
    """Synthetic plasma grids: Gaussian profile in x."""
    X = np.arange(nx) - nx // 2
    profile = np.exp(-(X**2) / (2 * (nx // 8)**2))
    ne = ne_peak * profile + 1e17  # cm⁻³
    zstar = Z_star * np.ones((nz, nx))
    ni = ne / zstar
    te = T_e * np.ones((nz, nx))
    return ne, ni, te, zstar

def check_output(label, detected, shape_2d=False):
    """Validate detector output."""
    assert detected is not None, f'{label}: None'
    assert np.isfinite(detected).all(), f'{label}: non-finite'
    if shape_2d:
        assert detected.ndim == 2, f'{label}: want 2D, got {detected.shape}'
        assert detected.shape[0] > 1 and detected.shape[1] > 1
    print(f'  \u2713 {label}: shape={detected.shape}, '
          f'min={detected.min():.4e}, max={detected.max():.4e}')

## Test 1: 2D Free-space Propagation

Point source → `propagate_analytically_2d` → FFT2 → frequency cutoff → IFFT2 → `square_and_downsample_2d`.  Also runs equivalent 1D for comparison.

In [ ]:
print('=' * 60)
print('Test 1: 2D Free-space propagation')
print('=' * 60)

# Use max_dx to satisfy Nyquist condition
nx, ny = 256, 256
N = nx * ny
wl = convert_energy_wavelength(8000.0)
z_det = 0.02
chunk = 4096
cutoff_angle = 0.015
cfreq = np.sin(cutoff_angle) / wl

dx = max_dx(z_det, 0.0, N, wl)
dy = dx
print(f'N={N}, dx={dx:.3e}, wl={wl:.3e}')

params_2d = SimParams(
    N=N, dx=dx, z_detector=z_det,
    detector_size=nx*dx,
    detector_pixel_size_x=dx*4,
    detector_pixel_size_y=dx*4,
    wl=wl, chunk_size=chunk,
    ny=ny, dy=dy,
    detector_size_x=nx*dx,
    detector_size_y=ny*dy,
)
print(f'2D mode: ny={params_2d.ny}, nx={params_2d.nx}, is_2d={params_2d.is_2d}')

source = PointSource(x=0.0, y=0.0, z=0.0)
u = NumpyVector(np.zeros(N, dtype=np.complex64))
U = NumpyVector(np.zeros(N, dtype=np.complex64))
source.propagate_to(z_det, params_2d, cfreq, u, U, None)

det_2d = square_and_downsample_2d(u, params_2d, z_det)
check_output('2D free-space', det_2d, shape_2d=True)

In [ ]:
# 1D reference (same dx)
params_1d = SimParams(
    N=N, dx=dx, z_detector=z_det,
    detector_size=nx*dx,
    detector_pixel_size_x=dx*4,
    detector_pixel_size_y=dx*4,
    wl=wl, chunk_size=chunk,
)
u1 = NumpyVector(np.zeros(N, dtype=np.complex64))
U1 = NumpyVector(np.zeros(N, dtype=np.complex64))
PointSource(x=0.0, z=0.0).propagate_to(z_det, params_1d, cfreq, u1, U1, None)
det_1d = square_and_downsample(u1, params_1d, z_det)
check_output('1D free-space (ref)', det_1d, shape_2d=False)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

im = axes[0].imshow(det_2d, cmap='inferno', origin='lower', aspect='auto')
axes[0].set_title('2D free-space intensity')
axes[0].set_xlabel('x (pix)'); axes[0].set_ylabel('y (pix)')
plt.colorbar(im, ax=axes[0])

mid = det_2d.shape[0] // 2
axes[1].plot(det_2d[mid, :], label='2D central row')
axes[1].plot(det_1d / det_1d.max() * det_2d[mid, :].max(),
             '--', label='1D (scaled)', alpha=0.7)
axes[1].set_title('Central cross-section')
axes[1].set_xlabel('x (pix)'); axes[1].set_ylabel('Intensity')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(rave_sim_dir / 'notebooks' / 'test_2d_simulation' / 'test1_freespace.png'), dpi=150)
plt.show()
print('\u2713 Test 1 complete')

## Test 2: 2D Sample Diffraction

3D material grid (W sphere in vacuum) → Sample._apply_2d() → Fresnel propagation → 2D detector image.  Compares against free-space reference.

In [ ]:
print('=' * 60)
print('Test 2: 2D Sample propagation')
print('=' * 60)

from optical_element import Sample, Material

nx, ny = 256, 256
N = nx * ny
wl = convert_energy_wavelength(8000.0)
z_sample = 0.005
z_det = 0.015
chunk = 4096
cutoff_freq = np.sin(0.025) / wl

dx = max_dx(z_sample, 0.0, N, wl)  # use source→sample distance for Nyquist
dy = dx
print(f'N={N}, dx={dx:.3e}')

params = SimParams(
    N=N, dx=dx, z_detector=z_det,
    detector_size=nx*dx,
    detector_pixel_size_x=dx*4,
    detector_pixel_size_y=dx*4,
    wl=wl, chunk_size=chunk,
    ny=ny, dy=dy,
    detector_size_x=nx*dx,
    detector_size_y=ny*dy,
)
print(f'2D mode: nx={params.nx}, ny={params.ny}')

# W sphere in vacuum
grid = make_sphere_grid(40, 40, 8, radius_px=15)
print(f'Grid shape: {grid.shape}, fill: {grid.mean():.3f}')

sample = Sample(
    z_start=z_sample,
    pixel_size_x=dx, pixel_size_y=dy, pixel_size_z=dx*2,
    grid=grid,
    materials=[Material('W', 19.35)],
    x_positions=np.array([0.0]),
    y_positions=np.array([0.0]),
)
sample.check_valid()

# Deltabeta lookup (with fallback)
try:
    from optical_element import generate_deltabeta_table, collect_all_materials
    mats = collect_all_materials([sample])
    dbt = generate_deltabeta_table(mats, convert_energy_wavelength(wl))
    sample.store_deltabetas(dbt)
    print('deltabeta from nist_lookup')
except Exception as e:
    print(f'nist_lookup failed ({e}), using manual W at 8 keV delta/beta')
    sample.db_list = np.array([0.0 + 0.0j, 4.5e-5 + 1.2e-5j], dtype=np.complex128)

In [ ]:
# Run: source → sample → detector

source = PointSource(x=0.0, y=0.0, z=0.0)
u = NumpyVector(np.zeros(N, dtype=np.complex64))
U = NumpyVector(np.zeros(N, dtype=np.complex64))

# source → sample
source.propagate_to(z_sample, params, cutoff_freq, u, U, None)

# apply sample (triggers _apply_2d since is_2d and grid.ndim == 3)
sample.apply(u, U, params, cutoff_freq, stepping_iteration=0, history=None)

# sample → detector
dz = z_det - (z_sample + sample.get_thickness())
propagate_2d(u, U, params.dx, params.get_dy(), params.wl,
             dz, params.chunk_size, cutoff_freq, params.nx, params.ny)

det_sample = square_and_downsample_2d(u, params, z_det)
check_output('2D sample', det_sample, shape_2d=True)

# free-space reference
u_ref = NumpyVector(np.zeros(N, dtype=np.complex64))
U_ref = NumpyVector(np.zeros(N, dtype=np.complex64))
source.propagate_to(z_det, params, cutoff_freq, u_ref, U_ref, None)
det_ref = square_and_downsample_2d(u_ref, params, z_det)

diff = np.abs(det_sample - det_ref)
print(f'Max difference from free-space: {diff.max():.4e}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
titles = ['With W sphere', 'Free space', '|Difference|']
for ax, img, t in zip(axes, [det_sample, det_ref, diff], titles):
    im = ax.imshow(img, cmap='inferno', origin='lower', aspect='auto')
    ax.set_title(t); ax.set_xlabel('x (pix)'); ax.set_ylabel('y (pix)')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig(str(rave_sim_dir / 'notebooks' / 'test_2d_simulation' / 'test2_sample.png'), dpi=150)
plt.show()
print('\u2713 Test 2 complete')

## Test 3: 1D PlasmaSample

Verifies plasma physics (delta_free > 0, beta > 0) using `plasma_delta_beta` directly.  PlasmaSample is currently 1D-only (uses `propagate()`, not `propagate_2d()`).

In [ ]:
print('=' * 60)
print('Test 3: PlasmaSample (1D) + plasma_delta_beta physics')
print('=' * 60)

# ── physics check ──────────────────────────────────────────
Z = 13                     # Al
Z_star = 8.0
T_e = 100.0                # eV
ne_pk = 5.0e21             # cm⁻³
energy = 8000.0            # eV

d, b, atlen = plasma_delta_beta(
    n_e=ne_pk, n_i=ne_pk / Z_star,
    T_e=T_e, Z_star=Z_star, Z=Z, energy=energy,
)
print(f'Plasma delta={d:.6e}, beta={b:.6e}, atlen={atlen:.4e} cm')
assert d > 0, 'delta_free should be positive'
assert b > 0, 'beta should be positive (absorption)'
assert np.isfinite(atlen)
print('  \u2713 Physics check passed')

In [ ]:
# Propagation with PlasmaSample
N = 65536
wl = convert_energy_wavelength(energy)
z_sample = 0.005
z_det = 0.02
det_pix = 1e-4
chunk = 4096

dx = max_dx(z_sample, 0.0, N, wl)
print(f'N={N}, dx={dx:.3e}')

params = SimParams(
    N=N, dx=dx, z_detector=z_det,
    detector_size=10e-3, detector_pixel_size_x=det_pix,
    detector_pixel_size_y=1.0,
    wl=wl, chunk_size=chunk,
)
print(f'1D mode: is_2d={params.is_2d}')

ne, ni, te, zs = make_gaussian_plasma(10, N // 8, ne_pk, Z_star, T_e)

plasma = PlasmaSample(
    z_start=z_sample, pixel_size_x=dx, pixel_size_z=dx*10,
    ne_grid=ne, ni_grid=ni, te_grid=te, zstar_grid=zs,
    Z=Z, x_positions=np.array([0.0]),
)
plasma.check_valid()
print(f'Plasma thickness: {plasma.get_thickness():.2e} m')

source = PointSource(x=0.0, z=0.0)
u = NumpyVector(np.zeros(N, dtype=np.complex64))
U = NumpyVector(np.zeros(N, dtype=np.complex64))
source.propagate_to(z_sample, params, 1000, u, U, None)
plasma.apply(u, U, params, 1000, stepping_iteration=0, history=None)
dz = z_det - (z_sample + plasma.get_thickness())
propagate(u, U, params.dx, params.wl, dz, params.chunk_size, 1000)
det_plasma = square_and_downsample(u, params, z_det)
check_output('Plasma 1D', det_plasma, shape_2d=False)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
x_det = (np.arange(len(det_plasma)) - len(det_plasma)/2) * det_pix * 1e6
ax.plot(x_det, det_plasma)
ax.set_xlabel('x (\u00b5m)'); ax.set_ylabel('Intensity')
ax.set_title('1D PlasmaSample intensity at detector')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(rave_sim_dir / 'notebooks' / 'test_2d_simulation' / 'test3_plasma_1d.png'), dpi=150)
plt.show()
print('\u2713 Test 3 complete')

## Test 4: Generate fast-wave Compatible 2D Config

Creates a simulation directory that can be run with the C++ GPU solver:
```bash
fastwave -s 0 <sim_path>
```

In [ ]:
print('=' * 60)
print('Test 4: Generate fast-wave 2D config')
print('=' * 60)

import multisim
import tempfile
import shutil
import yaml

nx, ny = 512, 512
dx = dy = 2.0e-7

# 2D (z x) grid
grid_2d = np.zeros((3, 40), dtype=np.uint32)
grid_2d[:, 15:25] = 1

tmpdir = Path(tempfile.mkdtemp(prefix='test2d_config_'))
grid_path = tmpdir / 'test_grid.npy'
np.save(grid_path, grid_2d)

config_dict = {
    'sim_params': {
        'N': nx * ny, 'nx': nx, 'ny': ny,
        'dx': dx, 'dy': dy,
        'z_detector': 0.3,
        'detector_size_x': 50e-6,         # 50 um
        'detector_size_y': 50e-6,
        'detector_pixel_size_x': 2e-6,
        'detector_pixel_size_y': 2e-6,
        'chunk_size': 256 * 1024 * 1024 // 16,
    },
    'use_disk_vector': False,
    'save_final_u_vectors': False,
    'dtype': 'c8',
    'multisource': {
        'type': 'points',
        'energy_range': [9950, 10050],
        'x_range': [-5e-7, 5e-7],       # +/- 0.5 um source
        'y_range': [-5e-7, 5e-7],
        'z': 0.0,
        'nr_source_points': 1,
        'seed': 42,
    },
    'elements': [{
        'type': 'sample',
        'z_start': 0.28,
        'pixel_size_x': 2e-7, 'pixel_size_y': 2e-7, 'pixel_size_z': 1e-6,
        'grid_path': str(grid_path),
        'materials': [['W', 19.35]],
        'x_positions': [0.0],
        'y_positions': [0.0],
    }],
}

outdir = tmpdir / 'sim_output'
outdir.mkdir(parents=True)

try:
    sim_path = multisim.setup_simulation(config_dict, tmpdir, outdir)
    print(f'Config generated at: {sim_path}')

    assert (sim_path / 'config.yaml').exists()
    assert (sim_path / 'computed.yaml').exists()
    subdirs = list(sim_path.glob('????????'))
    assert len(subdirs) == 1
    assert (subdirs[0] / 'subconfig.yaml').exists()

    with open(sim_path / 'config.yaml') as f:
        cfg = yaml.safe_load(f)
    sp = cfg['sim_params']
    assert sp.get('nx') == nx and sp.get('ny') == ny
    print(f'  Validated: N={sp["N"]}, nx={sp["nx"]}, ny={sp["ny"]}')

    print(f'\n  To run with C++: fastwave -s 0 {sim_path}')
finally:
    shutil.rmtree(tmpdir, ignore_errors=True)

print('  PASS')

## Summary: 2D Support Status

In [ ]:
print('''
\u2554\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2557\n\u2551              2D Support Summary                            \u2551\n\u2551  Python (big-wave)             C++ 3d variant (fast-wave)   \u2551\n\u2551  propagate_2d()         \u2705      propagate_2d CUDA    \u2705    \u2551\n\u2551  propagate_analytically_2d \u2705      (ballistic)         \u2705    \u2551\n\u2551  square_and_downsample_2d \u2705      (downsample)        \u2705    \u2551\n\u2551  frequency_cutoff_2d    \u2705      (circular mask)     \u2705    \u2551\n\u2551  Sample._apply_2d()     \u2705      apply_sample_2d()   \u2705    \u2551\n\u2551  PointSource y-support  \u2705      (y coordinate)      \u2705    \u2551\n\u2551  SimParams ny/dy/is_2d  \u2705      (nx/ny/is2d)        \u2705    \u2551\n\u2551  NumpyVector.fft2       \u2705      cuFFT 2D plan       \u2705    \u2551\n\u2551  wavesim 2D pipeline     \u2705      run_simulation_2d  \u2705    \u2551\n\u2551  NOT yet 2D:                                               \u2551\n\u2551    PlasmaSample     \u274c   (1D only both sides)           \u2551\n\u2551    precise_Sample   \u274c   (1D only both sides)           \u2551\n\u2551    Grating/EnvGrating \u274c   (1D only both sides)           \u2551\n\u255a\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u255d\n''')
print('All tests completed successfully.')